# Step 3.2 — Radar Multi-Frame Tracking (Upgraded, No UKF Yet) ✅

| | |
|---|---|
| **Input** | `output/step_2/radar/<sample>/<channel>.json` (from upgraded Step 2.2 — calibration already embedded per channel) |
| **Outputs** | `output/step_3/radar/track_<id>.json` — one file per track, points in GLOBAL frame |
| | `output/step_3/radar_tracking_summary.csv` |
| **Used by** | Step 5 (TTC estimation, radar-only baseline), Step 4 (fusion) |

---

### Bugs fixed — same family as Step 3.1, plus one extra

1. **`["points"]` unwrap fix** — same compatibility break as before, now against the upgraded Step 2.2 output.
2. **Per-channel calibration (radar-specific issue LiDAR doesn't have).** Radar has 5 physically separate sensors, each with its own sensor-to-ego calibration. The original code pooled raw local-frame `(x, y)` from all 5 channels together with no transform at all — a detection from `RADAR_FRONT` and one from `RADAR_BACK_LEFT` were being compared as if they shared a coordinate system, when they don't. Fixed by transforming **each channel's points using that channel's own calibration** before combining.
3. **Ego-motion contamination across frames** — same fix as Step 3.1: transform to the global frame so only true object motion affects tracking, not the car's own movement.
4. **Double-assignment bug** — same Hungarian algorithm fix as Step 3.1.
5. **No track termination** — same `MAX_MISSED_FRAMES` eviction as Step 3.1.

### Note
Radar calibration didn't need a separate lookup file this time — Step 2.2 already embeds each channel's `calibration` block directly in its output JSON, so this notebook reads it straight from there.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

import shutil
from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP0_DIR, STEP2_DIR, STEP3_DIR

RADAR_IN_DIR  = STEP2_DIR / "radar"
RADAR_OUT_DIR = STEP3_DIR / "radar"
if RADAR_OUT_DIR.exists():
    shutil.rmtree(RADAR_OUT_DIR)   # FIXED: clear stale per-object files before this run (was: accumulated silently across re-runs, never cleared)
RADAR_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RADAR_IN_DIR.exists():
    raise FileNotFoundError(f"Step 2.2 output not found at {RADAR_IN_DIR} — run Step 2.2 first.")

print(f"✅ RADAR_IN_DIR : {RADAR_IN_DIR}")
print(f"✅ RADAR_OUT_DIR: {RADAR_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ RADAR_IN_DIR : F:\Sensor fusion Research\output\step_2\radar
✅ RADAR_OUT_DIR: F:\Sensor fusion Research\output\step_3\radar


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Named constants (same values as Step 3.1 for consistency)
# ─────────────────────────────────────────────────────────────────

ASSOC_DIST_THRESHOLD = 3.0   # metres
MAX_MISSED_FRAMES    = 3     # frames

print(f"✅ ASSOC_DIST_THRESHOLD = {ASSOC_DIST_THRESHOLD}m, MAX_MISSED_FRAMES = {MAX_MISSED_FRAMES}")

✅ ASSOC_DIST_THRESHOLD = 3.0m, MAX_MISSED_FRAMES = 3


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Sensor-to-global transform
# Moved to src/geometry.py — shared with Step 1.1, Step 2.3.1, Step 3.1
# ─────────────────────────────────────────────────────────────────

import numpy as np
from pyquaternion import Quaternion
from src.geometry import transform_matrix, point_to_global

print("\u2705 Transform utilities loaded.")


✅ Transform utilities loaded.


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Tracker with one-to-one assignment + eviction (same as Step 3.1)
# FIXED: scene-boundary isolation — tracks never associate across a scene_name
#        change; at a boundary all active tracks are force-finalized and a
#        fresh track table starts for the new scene (no carried IDs/state).
# ─────────────────────────────────────────────────────────────────

import uuid
from scipy.optimize import linear_sum_assignment


class GlobalFrameTracker:
    def __init__(self, dist_thresh, max_missed):
        self.active_tracks = {}
        self.finished_tracks = {}
        self.dist_thresh = dist_thresh
        self.max_missed = max_missed
        self.current_scene_name = None
        self.association_log = []  # (prev_sample_id, prev_scene, curr_sample_id, curr_scene) per accepted match

    def update(self, detections_global, sample_id, timestamp, scene_name):
        if self.current_scene_name is not None and scene_name != self.current_scene_name:
            # Scene boundary: close out every active track from the previous scene and
            # start a fresh, empty track table. No track IDs, motion state, or "last known
            # position" carry across — each survivor is just finalized as-is.
            self.finished_tracks.update(self.active_tracks)
            self.active_tracks = {}
        self.current_scene_name = scene_name

        det_arr = np.array(detections_global) if detections_global else np.empty((0, 3))
        track_ids = list(self.active_tracks.keys())
        n_tracks, n_dets = len(track_ids), len(det_arr)

        matched_track_idx, matched_det_idx = set(), set()

        if n_tracks > 0 and n_dets > 0:
            cost = np.zeros((n_tracks, n_dets))
            for i, tid in enumerate(track_ids):
                pts = self.active_tracks[tid]["points"]      # [(sample_id, timestamp, [x,y,z]), ...]
                last_pos = np.array(pts[-1][2], dtype=float)
                pred = last_pos
                if len(pts) >= 2 and pts[-1][1] is not None and pts[-2][1] is not None:
                    dt_prev = (pts[-1][1] - pts[-2][1]) / 1e6
                    dt_now  = (timestamp   - pts[-1][1])  / 1e6
                    if dt_prev > 0 and dt_now > 0:
                        vel = (last_pos - np.array(pts[-2][2], dtype=float)) / dt_prev
                        spd = np.linalg.norm(vel)
                        if spd > 30.0:                        # clamp: 108 km/h, guards against
                            vel = vel / spd * 30.0             # one bad centroid flinging the gate
                        pred = last_pos + vel * dt_now
                cost[i] = np.linalg.norm(det_arr - pred, axis=1)

            row_idx, col_idx = linear_sum_assignment(cost)
            for r, c in zip(row_idx, col_idx):
                if cost[r, c] < self.dist_thresh:
                    tid = track_ids[r]
                    prev_sample_id, prev_timestamp, _ = self.active_tracks[tid]["points"][-1]
                    self.association_log.append((prev_sample_id, self.current_scene_name, sample_id, scene_name))
                    self.active_tracks[tid]["points"].append((sample_id, timestamp, detections_global[c]))
                    self.active_tracks[tid]["missed"] = 0
                    matched_track_idx.add(r)
                    matched_det_idx.add(c)

        for i, tid in enumerate(track_ids):
            if i not in matched_track_idx:
                self.active_tracks[tid]["missed"] += 1
                if self.active_tracks[tid]["missed"] > self.max_missed:
                    self.finished_tracks[tid] = self.active_tracks.pop(tid)

        for j in range(n_dets):
            if j not in matched_det_idx:
                tid = str(uuid.uuid4())[:8]
                self.active_tracks[tid] = {
                    "points": [(sample_id, timestamp, detections_global[j])],
                    "missed": 0
                }

    def save_tracks(self, out_dir):
        all_tracks = {**self.finished_tracks, **self.active_tracks}
        n_saved = 0
        for tid, track in all_tracks.items():
            if len(track["points"]) >= 2:
                with open(out_dir / f"track_{tid}.json", "w") as f:
                    json.dump(track["points"], f, indent=2)
                n_saved += 1
        return n_saved, len(all_tracks)


print("GlobalFrameTracker defined.")


GlobalFrameTracker defined.


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Main loop
# FIXED: unwraps ["points"]
# FIXED: each of the 5 radar channels transformed with ITS OWN calibration
#        before being combined into one per-sample detection list
# FIXED: one real object triggers 3-8 radar blips (echoes off different parts
#        of the same car); each was being tracked as a separate object. Now
#        DBSCAN groups blips within RADAR_CLUSTER_EPS of each other into one
#        detection (its centroid) before they reach the tracker.
# ─────────────────────────────────────────────────────────────────

import json
import numpy as np
from tqdm import tqdm
from sklearn.cluster import DBSCAN

RADAR_CLUSTER_EPS = 1.5   # metres — blips this close are treated as one object


def cluster_radar_detections(detections_global, eps):
    """Groups nearby blips (multiple radar echoes off the same car) into one
    detection per real object, using each cluster's centroid. min_samples=1
    so an isolated blip still becomes its own single-point detection."""
    if len(detections_global) <= 1:
        return detections_global
    pts = np.array(detections_global)
    labels = DBSCAN(eps=eps, min_samples=1).fit_predict(pts)
    return [pts[labels == lbl].mean(axis=0).tolist() for lbl in sorted(set(labels))]

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

tracker = GlobalFrameTracker(dist_thresh=ASSOC_DIST_THRESHOLD, max_missed=MAX_MISSED_FRAMES)
n_samples_processed = 0
n_samples_no_detections = 0

for sample_id in tqdm(sorted(samples_index.keys()), desc="Tracking radar objects"):
    sample_dir = RADAR_IN_DIR / sample_id
    if not sample_dir.exists():
        continue

    detections_global = []

    for radar_file in sample_dir.glob("RADAR_*.json"):
        with open(radar_file) as f:
            radar_data = json.load(f)

        points = radar_data["points"]              # FIXED — unwrap dict structure
        calibration = radar_data["calibration"]     # THIS channel's own calibration
        ego_pose = calibration["ego_pose"]
        calib = {
            "translation": calibration["sensor_to_ego_translation"],
            "rotation": calibration["sensor_to_ego_rotation"]
        }

        for pt in points:
            global_xyz = point_to_global(
                [pt["x"], pt["y"], pt.get("z", 0.0)], ego_pose, calib
            )
            detections_global.append(global_xyz.tolist())

    timestamp = samples_index[sample_id]["timestamp_us"]
    scene_name = samples_index[sample_id]["scene_name"]

    if len(detections_global) == 0:
        n_samples_no_detections += 1

    clustered_detections = cluster_radar_detections(detections_global, RADAR_CLUSTER_EPS)
    tracker.update(clustered_detections, sample_id, timestamp, scene_name)
    n_samples_processed += 1

n_saved, n_total = tracker.save_tracks(RADAR_OUT_DIR)

# ── Scene-boundary isolation check ─────────────────────────────────
cross_scene_associations = [a for a in tracker.association_log if a[1] != a[3]]
assert len(cross_scene_associations) == 0, \
    f"{len(cross_scene_associations)} cross-scene associations found: {cross_scene_associations[:5]}"

print(f"\nStep 3.2 complete.")
print(f"   Samples processed         : {n_samples_processed}")
print(f"   Samples with zero detections: {n_samples_no_detections}")
print(f"   Total tracks created       : {n_total}")
print(f"   Tracks saved (length >= 2)  : {n_saved}")
print(f"   Associations logged        : {len(tracker.association_log)} (0 cross-scene, verified)")
print(f"Saved to: {RADAR_OUT_DIR}")


Tracking radar objects:   0%|          | 0/404 [00:00<?, ?it/s]

Tracking radar objects:   0%|          | 1/404 [00:00<04:43,  1.42it/s]

Tracking radar objects:   0%|          | 2/404 [00:00<02:30,  2.67it/s]

Tracking radar objects:   1%|          | 5/404 [00:00<00:54,  7.36it/s]

Tracking radar objects:   2%|▏         | 8/404 [00:01<00:34, 11.37it/s]

...

Tracking radar objects:  97%|█████████▋| 390/404 [00:25<00:00, 32.00it/s]

Tracking radar objects:  98%|█████████▊| 394/404 [00:25<00:00, 16.95it/s]

Tracking radar objects:  99%|█████████▉| 400/404 [00:25<00:00, 23.37it/s]

Tracking radar objects: 100%|██████████| 404/404 [00:25<00:00, 15.70it/s]


Step 3.2 complete.
   Samples processed         : 404
   Samples with zero detections: 3
   Total tracks created       : 4025
   Tracks saved (length >= 2)  : 1693
   Associations logged        : 3797 (0 cross-scene, verified)
Saved to: F:\Sensor fusion Research\output\step_3\radar


In [6]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 — Track length distribution — same sanity check as Step 3.1
# ─────────────────────────────────────────────────────────────────

import pandas as pd

track_lengths = []
for track_file in RADAR_OUT_DIR.glob("track_*.json"):
    with open(track_file) as f:
        track = json.load(f)
    track_lengths.append(len(track))

length_df = pd.DataFrame({"track_length": track_lengths})
summary_path = STEP3_DIR / "radar_tracking_summary.csv"
length_df.to_csv(summary_path, index=False)

print(f"✅ Summary saved: {summary_path}")
print(f"   Total tracks       : {len(length_df)}")
print(f"   Mean track length  : {length_df['track_length'].mean():.1f} frames")
print(f"   Median track length: {length_df['track_length'].median():.0f} frames")
print(f"   Tracks of length 2 (most fragmented): "
      f"{(length_df['track_length'] == 2).sum()} ({(length_df['track_length'] == 2).mean()*100:.1f}%)")
print(f"   Tracks of length 10+: {(length_df['track_length'] >= 10).sum()}")

display(length_df.describe())

✅ Summary saved: F:\Sensor fusion Research\output\step_3\radar_tracking_summary.csv
   Total tracks       : 1693
   Mean track length  : 3.2 frames
   Median track length: 2 frames
   Tracks of length 2 (most fragmented): 878 (51.9%)
   Tracks of length 10+: 45


,track_length
count,1693.000000
mean,3.242764
std,2.285376
min,2.000000
25%,2.000000
50%,2.000000
75%,4.000000
max,34.000000
